In [0]:
-- Create the schema
USE CATALOG cps714;
CREATE SCHEMA IF NOT EXISTS restaurant_manager;

USE SCHEMA restaurant_manager;

-- ── Restaurant ──────────────────────────────────────────────
-- Top-level entity. One restaurant owns many tables.
CREATE TABLE IF NOT EXISTS restaurant (
    restaurant_id   BIGINT       GENERATED ALWAYS AS IDENTITY,
    name             STRING       NOT NULL,
    address          STRING,
    phone            STRING,
    created_at       TIMESTAMP    DEFAULT CURRENT_TIMESTAMP,
    CONSTRAINT pk_restaurant PRIMARY KEY (restaurant_id)
)
TBLPROPERTIES('delta.feature.allowColumnDefaults' = 'supported');

-- ── Table (dining table inside a restaurant) ────────────────
-- Named `restaurant_table` to avoid the reserved keyword `table`.
CREATE TABLE IF NOT EXISTS restaurant_table (
    table_id         BIGINT       GENERATED ALWAYS AS IDENTITY,
    restaurant_id    BIGINT       NOT NULL,
    table_number     INT          NOT NULL,   -- logical label shown to staff
    capacity         INT          NOT NULL DEFAULT 4,  -- number of physical seats
    status           STRING       DEFAULT 'available',  -- available | occupied | reserved
    CONSTRAINT pk_restaurant_table PRIMARY KEY (table_id),
    CONSTRAINT fk_table_restaurant FOREIGN KEY (restaurant_id)
        REFERENCES restaurant (restaurant_id)
)
TBLPROPERTIES('delta.feature.allowColumnDefaults' = 'supported');

-- ── Seat (an individual seat at a table) ───────────────────
CREATE TABLE IF NOT EXISTS seat (
    seat_id          BIGINT       GENERATED ALWAYS AS IDENTITY,
    table_id         BIGINT       NOT NULL,
    seat_number      INT          NOT NULL,   -- position at the table
    CONSTRAINT pk_seat PRIMARY KEY (seat_id),
    CONSTRAINT fk_seat_table FOREIGN KEY (table_id)
        REFERENCES restaurant_table (table_id)
);

-- ── Menu Item (items the restaurant offers) ────────────────
CREATE TABLE IF NOT EXISTS menu_item (
    menu_item_id     BIGINT       GENERATED ALWAYS AS IDENTITY,
    restaurant_id    BIGINT       NOT NULL,
    name             STRING       NOT NULL,
    description      STRING,
    price            DECIMAL(8,2) NOT NULL,
    category         STRING,   -- appetizer | main | dessert | beverage | …
    available        BOOLEAN      DEFAULT TRUE,
    CONSTRAINT pk_menu_item PRIMARY KEY (menu_item_id),
    CONSTRAINT fk_menu_item_restaurant FOREIGN KEY (restaurant_id)
        REFERENCES restaurant (restaurant_id)
)
TBLPROPERTIES('delta.feature.allowColumnDefaults' = 'supported');

-- ── Seat Order Item (menu items ordered at a seat) ──────────
-- Junction table: each row is a menu item ordered by a specific seat.
CREATE TABLE IF NOT EXISTS seat_order_item (
    seat_order_id    BIGINT       GENERATED ALWAYS AS IDENTITY,
    seat_id          BIGINT       NOT NULL,
    menu_item_id     BIGINT       NOT NULL,
    quantity         INT          NOT NULL DEFAULT 1,
    order_time       TIMESTAMP    DEFAULT CURRENT_TIMESTAMP,
    status           STRING       DEFAULT 'pending',  -- pending | preparing | served
    CONSTRAINT pk_seat_order_item PRIMARY KEY (seat_order_id),
    CONSTRAINT fk_seat_order_item_seat FOREIGN KEY (seat_id)
        REFERENCES seat (seat_id),
    CONSTRAINT fk_seat_order_item_menu FOREIGN KEY (menu_item_id)
        REFERENCES menu_item (menu_item_id)
)
TBLPROPERTIES('delta.feature.allowColumnDefaults' = 'supported');

-- ── Employee (staff working at a restaurant) ──────────────
CREATE TABLE IF NOT EXISTS employee (
    employee_id     BIGINT       GENERATED ALWAYS AS IDENTITY,
    restaurant_id   BIGINT       NOT NULL,
    first_name       STRING       NOT NULL,
    last_name        STRING       NOT NULL,
    role             STRING       NOT NULL DEFAULT 'server',  -- manager | chef | server | host | bartender | …
    phone            STRING,
    hire_date        DATE,
    status           STRING       DEFAULT 'active',  -- active | inactive
    CONSTRAINT pk_employee PRIMARY KEY (employee_id),
    CONSTRAINT fk_employee_restaurant FOREIGN KEY (restaurant_id)
        REFERENCES restaurant (restaurant_id)
)
TBLPROPERTIES('delta.feature.allowColumnDefaults' = 'supported');

-- ── Helpful indexes ─────────────────────────────────────────
-- Note: CREATE INDEX is not supported in Databricks SQL / Delta Lake.
-- Delta Lake handles data skipping automatically. For explicit clustering,
-- use CLUSTER BY in CREATE TABLE or OPTIMIZE ... ZORDER BY.
-- CREATE INDEX IF NOT EXISTS idx_table_restaurant ON restaurant_table (restaurant_id);
-- CREATE INDEX IF NOT EXISTS idx_seat_table      ON seat (table_id);
-- CREATE INDEX IF NOT EXISTS idx_menu_restaurant  ON menu_item (restaurant_id);
-- CREATE INDEX IF NOT EXISTS idx_order_seat       ON seat_order_item (seat_id);